# Tutorial 12 — Kafka reliability and event-time correctness

**Goal.** Validate the Avro contracts, publish a bounded stream, inject logical delivery faults, and measure what a consumer can recover.

**Prerequisites.** Base install for the offline harness; `poetry install -E kafka` and Docker for a real broker. The harness never drops TCP packets—it models logical Kafka delivery semantics.

**Produces.** Sent/acknowledged/dropped/retried/duplicated/late/reordered/deduplicated counts, partition skew, and an auditable manifest.


In [ ]:
from pathlib import Path

from fraudtwin import generate
from fraudtwin.config import load_config
from fraudtwin.kafka import publication_records
from fraudtwin.kafka_chaos import KafkaChaosConfig, simulate_delivery

root = Path.cwd()
config = load_config(root / "configs" / "minimal.yaml")
run = generate(config, write=True, output_dir=root / "runs" / "tutorial-12")
records = publication_records(run.behavior, run.manifest.run_id)
print({"payment_events": len(run.behavior.payment_events), "all_contract_records": len(records)})

## Contracts and topic options

The bundled registry is authoritative. In a broker deployment use a topic prefix, stable `payment_id` key, idempotent producer, `acks=all`, bounded retries, and explicit pacing (`batch`, real-time, or accelerated). Schema compatibility failures must reject the message rather than mutate its identity.


In [ ]:
from fraudtwin.contracts import contract_registry

registry = contract_registry()
print(registry.validate())
result = simulate_delivery(
    records,
    KafkaChaosConfig(
        seed=7,
        boundary="producer",
        drop_probability=0.02,
        duplicate_probability=0.03,
        retry_probability=0.05,
        max_delay_seconds=30,
        reorder_window=100,
        partition_count=6,
        partition_skew_probability=0.2,
    ),
)
print(result.manifest)

## Run against Kafka (optional)

Start `docker compose --profile streaming up -d`, set `FRAUDTWIN_KAFKA_BOOTSTRAP_SERVERS` and `FRAUDTWIN_SCHEMA_REGISTRY_URL`, then run `fraudtwin generate ...` with `outputs.kafka: true`. The CLI harness can replay a generated run:

```console
fraudtwin kafka chaos --run-id <run-id> --boundary producer --drop-rate 0.02 --duplicate-rate 0.03 --retry-rate 0.05 --delay-seconds 30 --reorder-window 100 --output-dir runs
```


In [ ]:
import json

assert result.input_fingerprint
assert result.emitted_count == sum(result.partition_counts.values())
assert all(envelope.event_id == envelope.record_id for envelope in result.envelopes)
print(
    json.dumps(
        {
            "lost": result.dropped_count,
            "duplicates": result.deduplicated_count,
            "late": result.late_count,
            "reordered": result.out_of_order_count,
            "output_fingerprint": result.output_fingerprint,
        },
        indent=2,
    )
)

Use stable event identity for consumer deduplication and event-time watermarks, not transport message IDs. Next: [Tutorial 13 — data-quality incident response and replay](13-operational-lakehouse-observability.ipynb) and the [Kafka reliability guide](../kafka-reliability.md).
